# Prepare K-Hairstyle Attribute Dataset

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

from systems.static_auto_tryon.auto_app.ml.datasets import (
    RAW_KHAIRSTYLE_IMAGE_DIR,
    RAW_KHAIRSTYLE_LABEL_DIR,
    build_attribute_records_from_khairstyle,
    build_label_vocab,
    train_val_split,
    write_jsonl_manifest,
)
from systems.static_auto_tryon.auto_app.ml.khairstyle_translation import (
    FIELD_TRANSLATIONS,
    build_normalized_attributes,
    repair_mojibake,
    translate_labels,
)

RAW_LABEL_ROOT = RAW_KHAIRSTYLE_LABEL_DIR
RAW_IMAGE_ROOT = RAW_KHAIRSTYLE_IMAGE_DIR
DATASET_DIR = BACKEND_ROOT / 'data' / 'datasets' / 'hairstyle_attribute'

print('Project root:', PROJECT_ROOT)
print('Raw label root exists:', RAW_LABEL_ROOT.exists())
print('Raw image root exists:', RAW_IMAGE_ROOT.exists())


In [ ]:
pd.Series({field: len(values) for field, values in FIELD_TRANSLATIONS.items()}).sort_values(ascending=False)


In [ ]:
raw_json_paths = sorted(RAW_LABEL_ROOT.rglob('*.json'))
sample_raw_path = raw_json_paths[0]
sample_raw_payload = json.loads(sample_raw_path.read_text(encoding='utf-8', errors='replace'))

sample_fields = ['basestyle', 'basestyle-type', 'length', 'curl', 'bang', 'side', 'color', 'partition', 'sex']
pd.DataFrame(
    {
        'raw_value': [sample_raw_payload.get(field) for field in sample_fields],
        'repaired_value': [repair_mojibake(sample_raw_payload.get(field)) for field in sample_fields],
    },
    index=sample_fields,
)


In [ ]:
sample_translated = translate_labels(sample_raw_payload)
sample_normalized = build_normalized_attributes(sample_translated)

display(pd.Series(sample_translated, name='translated_labels'))
display(pd.Series(sample_normalized, name='normalized_attributes'))


In [ ]:
coverage_rows = []
for path in raw_json_paths[:500]:
    payload = json.loads(path.read_text(encoding='utf-8', errors='replace'))
    translated = translate_labels(payload)
    coverage_rows.append(
        {
            'path': str(path.relative_to(PROJECT_ROOT)).replace('\\', '/'),
            'unmapped_fields': [field for field, value in translated.items() if value.startswith('unmapped_')],
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
coverage_df['unmapped_count'] = coverage_df['unmapped_fields'].apply(len)
coverage_df['unmapped_count'].value_counts().sort_index()


In [ ]:
coverage_df[coverage_df['unmapped_count'] > 0].head(20)


In [ ]:
MAX_RECORDS = None
TRAIN_RATIO = 0.85
RANDOM_SEED = 42
TRAIN_MANIFEST = DATASET_DIR / 'train.jsonl'
VAL_MANIFEST = DATASET_DIR / 'val.jsonl'
VOCAB_PATH = DATASET_DIR / 'label_vocab.json'
SUMMARY_PATH = DATASET_DIR / 'summary.json'


In [ ]:
records = build_attribute_records_from_khairstyle(limit=MAX_RECORDS)
train_records, val_records = train_val_split(records, train_ratio=TRAIN_RATIO, seed=RANDOM_SEED)
label_vocab = build_label_vocab(records)

write_jsonl_manifest(train_records, TRAIN_MANIFEST)
write_jsonl_manifest(val_records, VAL_MANIFEST)
VOCAB_PATH.parent.mkdir(parents=True, exist_ok=True)
VOCAB_PATH.write_text(json.dumps(label_vocab, indent=2, ensure_ascii=False), encoding='utf-8')

summary = {
    'total_records': len(records),
    'train_records': len(train_records),
    'val_records': len(val_records),
    'label_fields': list(label_vocab.keys()),
    'label_class_counts': {field: len(values) for field, values in label_vocab.items()},
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
pd.Series(summary)


In [ ]:
pd.DataFrame(records[:5])
